# notebook_01_oracle_to_databricks_migration
### Oracle ➜ Databricks (Spark JDBC) ➜ Delta (Bronze)

This notebook performs a **full-load migration** of an Oracle table into a **Delta table** in Databricks.

## What you’ll do
1. Configure connection parameters (widgets)
2. Read Oracle metadata (optional)
3. Choose a partition strategy for parallel JDBC reads
4. Read from Oracle (with parallelism)
5. Add ingestion metadata columns
6. Write to Delta (Bronze)
7. Validate row counts (and optional simple checksum)

> Notes:
- Ensure Oracle JDBC driver (ojdbc) is installed on the cluster.
- Ensure network connectivity (VPC/VNet peering/VPN, firewall rules, listener).
- Use Databricks **Secrets** for credentials (recommended).


## 1) Parameters (widgets)
Set these once and rerun the notebook. In Databricks, widgets appear at the top of the notebook.

In [ ]:
# Databricks widgets for easy parameterization
dbutils.widgets.text("oracle_host", "myoracle.company.com")
dbutils.widgets.text("oracle_port", "1521")
dbutils.widgets.text("oracle_service", "ORCLPDB1")  # service name (common), not SID
dbutils.widgets.text("oracle_schema", "SALES")
dbutils.widgets.text("table_name", "ORDERS")

# Partitioning (recommended for larger tables)
dbutils.widgets.text("partition_col", "ORDER_ID")
dbutils.widgets.text("num_partitions", "16")

# Target naming
dbutils.widgets.text("target_catalog", "")   # optional: Unity Catalog, e.g. "main"
dbutils.widgets.text("target_schema", "bronze_oracle")
dbutils.widgets.text("target_table_prefix", "ora_")

oracle_host = dbutils.widgets.get("oracle_host")
oracle_port = dbutils.widgets.get("oracle_port")
oracle_service = dbutils.widgets.get("oracle_service")
oracle_schema = dbutils.widgets.get("oracle_schema")
table_name = dbutils.widgets.get("table_name")

partition_col = dbutils.widgets.get("partition_col")
num_partitions = int(dbutils.widgets.get("num_partitions"))

target_catalog = dbutils.widgets.get("target_catalog")
target_schema = dbutils.widgets.get("target_schema")
target_table_prefix = dbutils.widgets.get("target_table_prefix")

full_oracle_table = f"{oracle_schema}.{table_name}"

if target_catalog.strip():
    target_fqn = f"{target_catalog}.{target_schema}.{target_table_prefix}{table_name.lower()}"
else:
    target_fqn = f"{target_schema}.{target_table_prefix}{table_name.lower()}"

print("Source:", full_oracle_table)
print("Target:", target_fqn)


## 2) Credentials via Databricks Secrets
Create a secret scope (example: `oracle-secrets`) and store keys `username` and `password`.

- In the Databricks UI: **Secrets** or via CLI.
- Replace the scope/key names below if needed.

> If you don't have secrets set up yet, you can temporarily hardcode credentials for a demo, but **do not commit secrets to GitHub**.

In [ ]:
# Recommended: fetch credentials from Databricks Secrets
# Update scope/key names to match your environment.
oracle_user = dbutils.secrets.get("oracle-secrets", "username")
oracle_pwd  = dbutils.secrets.get("oracle-secrets", "password")

# Build JDBC URL (service name variant)
jdbc_url = f"jdbc:oracle:thin:@//{oracle_host}:{oracle_port}/{oracle_service}"

connection_props = {
    "user": oracle_user,
    "password": oracle_pwd,
    "driver": "oracle.jdbc.OracleDriver",
    # Optional tuning:
    "fetchsize": "10000"
}

print("JDBC URL:", jdbc_url)


## 3) (Optional) Read Oracle table metadata
This step helps detect tricky types (CLOB/BLOB) and supports selecting columns explicitly.

In [ ]:
metadata_query = f"""(
SELECT COLUMN_NAME, DATA_TYPE, DATA_LENGTH, DATA_PRECISION, DATA_SCALE, NULLABLE
FROM ALL_TAB_COLUMNS
WHERE OWNER = '{oracle_schema.upper()}'
  AND TABLE_NAME = '{table_name.upper()}'
ORDER BY COLUMN_ID
)"""


meta_df = (spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", metadata_query)
    .options(**connection_props)
    .load()
)

display(meta_df)


## 4) Compute partition bounds for parallel JDBC read
For best performance on large tables, use JDBC partitioning.

You need:
- `partitionColumn` (numeric/high-cardinality)
- `lowerBound` and `upperBound`
- `numPartitions`

If you cannot find a good numeric column, use a surrogate (e.g., hash buckets) or perform a non-partitioned read for smaller tables.

In [ ]:
bounds_query = f"""(SELECT MIN({partition_col}) AS MIN_V, MAX({partition_col}) AS MAX_V
FROM {full_oracle_table})"""

bounds_row = (spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", bounds_query)
    .options(**connection_props)
    .load()
    .collect()[0]
)

lower = int(bounds_row["MIN_V"])
upper = int(bounds_row["MAX_V"])

print("Partition column:", partition_col)
print("Lower bound:", lower)
print("Upper bound:", upper)
print("Num partitions:", num_partitions)


## 5) Read from Oracle (parallel JDBC)
For small tables, you can remove the partition options and do a simple read.

### Handling CLOB/BLOB
Spark JDBC can struggle with LOB columns. For a first migration demo, exclude them by selecting only non-LOB columns.

Below we auto-exclude `CLOB`, `BLOB`, `NCLOB` if metadata is available. If not, fallback to reading the full table.

In [ ]:
from pyspark.sql import functions as F

# Try to exclude LOB columns using metadata (if meta_df exists)
lob_types = {"CLOB", "BLOB", "NCLOB"}
cols = None

try:
    meta_rows = meta_df.select("COLUMN_NAME", "DATA_TYPE").collect()
    non_lob_cols = [r["COLUMN_NAME"] for r in meta_rows if (r["DATA_TYPE"] or "").upper() not in lob_types]
    if len(non_lob_cols) > 0:
        cols = non_lob_cols
except Exception as e:
    print("Metadata not available or failed; proceeding without column pruning.")
    cols = None

if cols:
    select_list = ", ".join([f'"{c}"' for c in cols])  # quoted identifiers
    dbtable_opt = f"(SELECT {select_list} FROM {full_oracle_table}) t"
    print(f"Reading with {len(cols)} columns (LOB excluded).")
else:
    dbtable_opt = full_oracle_table
    print("Reading full table (no column pruning).")

src_df = (spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", dbtable_opt)
    .option("partitionColumn", partition_col)
    .option("lowerBound", str(lower))
    .option("upperBound", str(upper))
    .option("numPartitions", str(num_partitions))
    .options(**connection_props)
    .load()
)

display(src_df.limit(10))
print("Source rowcount (Spark):", src_df.count())


## 6) Add ingestion metadata columns (Bronze best practice)
These columns help with lineage, debugging, and incremental patterns later.

In [ ]:
bronze_df = (src_df
    .withColumn("_ingest_ts", F.current_timestamp())
    .withColumn("_source_system", F.lit("oracle"))
    .withColumn("_source_table", F.lit(full_oracle_table))
)

display(bronze_df.limit(10))


## 7) Create target schema and write to Delta
- Uses `overwrite` for initial full load.
- Sets `overwriteSchema=true` to align schema.

> In production, you may write to an external location and then register the table, or use a controlled migration framework.

In [ ]:
# Create target database/schema
if target_catalog.strip():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{target_schema}")
else:
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {target_schema}")

# Write to Delta (initial full load)
(bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_fqn)
)

print("Wrote Delta table:", target_fqn)
print("Delta rowcount:", spark.table(target_fqn).count())


## 8) Validation / Reconciliation
At minimum, compare row counts between Oracle and Delta.
For additional confidence, compare simple aggregations (e.g., SUM of numeric PK).

In [ ]:
# Oracle COUNT(*) pushdown
oracle_count_query = f"(SELECT COUNT(*) AS CNT FROM {full_oracle_table}) t"

oracle_cnt = (spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", oracle_count_query)
    .options(**connection_props)
    .load()
    .collect()[0]["CNT"]
)

delta_cnt = spark.table(target_fqn).count()

print("Oracle count:", oracle_cnt)
print("Delta count :", delta_cnt)
print("Match?      :", oracle_cnt == delta_cnt)


In [ ]:
# Optional simple checksum using SUM(partition_col) for numeric partition columns
checksum_query = f"(SELECT COUNT(*) AS CNT, SUM({partition_col}) AS SUM_V FROM {full_oracle_table}) t"

oracle_stats = (spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", checksum_query)
    .options(**connection_props)
    .load()
    .collect()[0]
)

delta_stats = spark.table(target_fqn).agg(
    F.count("*").alias("CNT"),
    F.sum(F.col(partition_col)).alias("SUM_V")
).collect()[0]

print("Oracle stats:", dict(oracle_stats.asDict()))
print("Delta stats :", dict(delta_stats.asDict()))
print("Checksum match?", oracle_stats["CNT"] == delta_stats["CNT"] and oracle_stats["SUM_V"] == delta_stats["SUM_V"])


## 9) Optional: Optimize Delta table
Run `OPTIMIZE` on large tables and consider `ZORDER` on frequently-filtered columns.
Skip this on small demo tables.

In [ ]:
# Optional: uncomment for large tables
# spark.sql(f"OPTIMIZE {target_fqn}")
# spark.sql(f"OPTIMIZE {target_fqn} ZORDER BY ({partition_col})")


## Next notebook
- **notebook_02_oracle_cdc_incremental_merge**: add incremental ingestion (watermark/CDC) using `MERGE INTO`.
- Add logging to a control table (ingestion run id, counts, durations, status).
